In [1]:
nvars = 4 # number of variables
BR = QQ[",".join("x"+str(i) for i in range(1, nvars+1))+",z"] # polynomial ring in x1, ..., xnvars, z
# BR is a global variable
BR.inject_variables() # make it so you can use those variables
SymmetricFunctions(QQ).inject_shorthands(verbose=False) # define the bases of symmetric functions

def do_P_k(k, n):
    # we are computing p_k[s_n] = s_n[p_k]
    # evaluate at the generators gens = BR.gens()[:-1] = (x1, x2, x3, x4)
    global BR,nvars
    CR = s[[]].expand(nvars).parent()
    return s[n].expand(nvars).subs({CR('x'+str(i)) : BR('x'+str(i+1))**k for i in range(nvars)})
@cached_function
def do_P_lambda(la, n):
    global BR,nvars
    a=expand(mul(do_P_k(p, n) for p in la)*mul(BR('x'+str(i+1))-BR('x'+str(j+1)) for i in range(nvars) for j in range(i+1, nvars)))
    return sum(c*mul(BR('x'+str(i+1))^(v[i]-(nvars-i-1)) for i in range(nvars))\
               for (v,c) in a.monomial_coefficients().items() if all(v[i]>v[i+1] for i in range(nvars-1)))
@cached_function
def den_coeff(d):
    global BR
    gz = BR.gens()[-1] # the last variable is the z
    return BR(SR(den_guess()).coefficient(gz,d))
def calc_num(la, d):
    return sum(den_coeff(d-r)*do_P_lambda(Partition(la),r) for r in range(d+1))

"""
This is the denominator that you are trying to determine
"""
@cached_function
def den_guess():
    global BR
    m1 = z*x1**5
    m2 = z*x1**4*x2
    m3 = z**2*x1**5*x2**5
    m4 = z**8*x1**20*x2**20
    m5 = z**4*x1**12*x2**4*x3**4
    m6 = z**2*x1**4*x2**4*x3**2
    m7 = z**3*x1**5*x2**5*x3**5
    m8 = -z**6*x1**10*x2**10*x3**10
    m9 = z*x1**2*x2*x3*x4
    m10 = -z**2*x1**4*x2**2*x3**2*x4**2
    m11 = -z**4*x1**5*x2**5*x3**5*x4**5
    m12 = z**3*x1**4*x2**4*x3**4*x4**3
    m13 = z**8*x1**12*x2**12*x3**8*x4**8
    return BR((1-m1)*(1-m2)*(1-m3)*(1-m4)*(1-m5)*(1-m6)*(1-m7)*(1-m8)*(1-m9)*(1-m10)*(1-m11)*(1-m12)*(1-m13))

Defining x1, x2, x3, x4, z


In [2]:
'''
#Version that works with older versions of Sage
@cached_function
def do_P_lambda(la, n):
    global BR, nvars
    a = expand(mul(do_P_k(p, n) for p in la) * mul(BR('x'+str(i+1)) - BR('x'+str(j+1)) for i in range(nvars) for j in range(i+1, nvars)))
    
    # Change is on the line below: swap .monomial_coefficients() for .dict()
    return sum(c * mul(BR('x'+str(i+1))^(v[i] - (nvars-i-1)) for i in range(nvars)) \
               for (v,c) in a.dict().items() if all(v[i] > v[i+1] for i in range(nvars-1)))
'''

"\n#Version that works with older versions of Sage\n@cached_function\ndef do_P_lambda(la, n):\n    global BR, nvars\n    a = expand(mul(do_P_k(p, n) for p in la) * mul(BR('x'+str(i+1)) - BR('x'+str(j+1)) for i in range(nvars) for j in range(i+1, nvars)))\n\n    # Change is on the line below: swap .monomial_coefficients() for .dict()\n    return sum(c * mul(BR('x'+str(i+1))^(v[i] - (nvars-i-1)) for i in range(nvars))                for (v,c) in a.dict().items() if all(v[i] > v[i+1] for i in range(nvars-1)))\n"

In [3]:
out = 0
for d in range(0, 300):
    CC = calc_num([4,1], d)
    if CC:
        CC_list = list(CC)
        if len(CC_list) > 6:
            front = CC_list[:3]
            back = CC_list[-3:]
            print(d, len(CC_list), "FRONT:", front, "BACK:", back)
        else:
            print(d, len(CC_list), CC_list)
    else:
        print(d, 0, "[]")
    
    out += z**d * CC

0 1 [(1, 1)]
1 4 [(-1, x1^4*x2), (-1, x1^3*x2^2), (1, x1^2*x2^2*x3), (-1, x1^2*x2*x3*x4)]
2 10 FRONT: [(1, x1^8*x2^2), (1, x1^6*x2^4), (-1, x1^5*x2^5)] BACK: [(-1, x1^4*x2^3*x3^2*x4), (1, x1^3*x2^3*x3^3*x4), (1, x1^4*x2^2*x3^2*x4^2)]
3 21 FRONT: [(-1, x1^10*x2^5), (1, x1^9*x2^6), (1, x1^8*x2^7)] BACK: [(1, x1^7*x2^3*x3^3*x4^2), (-1, x1^6*x2^3*x3^3*x4^3), (-1, x1^4*x2^4*x3^4*x4^3)]
4 44 FRONT: [(-1, x1^13*x2^7), (1, x1^12*x2^8), (-2, x1^11*x2^9)] BACK: [(1, x1^8*x2^4*x3^4*x4^4), (-1, x1^7*x2^5*x3^4*x4^4), (2, x1^6*x2^5*x3^5*x4^4)]
5 74 FRONT: [(1, x1^15*x2^10), (1, x1^13*x2^12), (-2, x1^15*x2^9*x3)] BACK: [(-1, x1^8*x2^7*x3^5*x4^5), (-2, x1^8*x2^6*x3^6*x4^5), (1, x1^7*x2^7*x3^6*x4^5)]
6 115 FRONT: [(-2, x1^17*x2^13), (1, x1^16*x2^14), (-1, x1^15*x2^15)] BACK: [(2, x1^10*x2^7*x3^7*x4^6), (-1, x1^9*x2^8*x3^7*x4^6), (1, x1^8*x2^8*x3^8*x4^6)]
7 173 FRONT: [(1, x1^20*x2^15), (1, x1^19*x2^16), (-1, x1^18*x2^17)] BACK: [(-1, x1^12*x2^8*x3^8*x4^7), (1, x1^11*x2^9*x3^8*x4^7), (-1, x1^10*x2^9*x3^

KeyboardInterrupt: 

In [6]:
out=0
for d in range(0,300):
    CC = calc_num([4,1],d)
    print(d,len(list(CC)), list(CC)[:3])
    # d is the degree
    # len(list(CC)) is how "big" the expression is at degree d
    # list(CC)[:3] gives three terms of the set of all terms in the expression
    #   (you can try replacing this with list(CC)[-3:] if you don't see patterns)
    print("*************")
    out+=z**d*CC

0 1 [(1, 1)]
*************
1 4 [(-1, x1^4*x2), (-1, x1^3*x2^2), (1, x1^2*x2^2*x3)]
*************
2 10 [(1, x1^8*x2^2), (1, x1^6*x2^4), (-1, x1^5*x2^5)]
*************
3 20 [(-1, x1^10*x2^5), (1, x1^9*x2^6), (1, x1^8*x2^7)]
*************
4 42 [(-1, x1^13*x2^7), (1, x1^12*x2^8), (-2, x1^11*x2^9)]
*************
5 73 [(1, x1^15*x2^10), (1, x1^13*x2^12), (-2, x1^15*x2^9*x3)]
*************
6 117 [(-2, x1^17*x2^13), (1, x1^16*x2^14), (-1, x1^15*x2^15)]
*************
7 172 [(1, x1^20*x2^15), (1, x1^19*x2^16), (-1, x1^18*x2^17)]
*************
8 244 [(-1, x1^23*x2^17), (1, x1^22*x2^18), (1, x1^20*x2^20)]
*************
9 332 [(-1, x1^25*x2^20), (-1, x1^24*x2^21), (2, x1^24*x2^20*x3)]
*************
10 421 [(1, x1^28*x2^22), (1, x1^25*x2^24*x3), (1, x1^27*x2^21*x3^2)]
*************
11 537 [(-1, x1^29*x2^25*x3), (1, x1^29*x2^24*x3^2), (1, x1^28*x2^25*x3^2)]
*************
12 637 [(-1, x1^32*x2^26*x3^2), (-1, x1^29*x2^28*x3^3), (1, x1^35*x2^21*x3^4)]
*************
13 764 [(1, x1^33*x2^29*x3^3), (1, x1^

KeyboardInterrupt: 

In [4]:
factor(out)

(-1) * (x1^3*x2*x3*z - 1) * (x1^3*x2^3*x3^2*x4^2*z^2 + 1) * (x1^5*x2^5*z^2 - 1)^2 * (x1^68*x2^53*x3^34*x4^15*z^34 - x1^66*x2^52*x3^33*x4^14*z^33 + x1^66*x2^51*x3^33*x4^15*z^33 + x1^65*x2^52*x3^33*x4^15*z^33 - x1^65*x2^51*x3^34*x4^15*z^33 - x1^64*x2^52*x3^34*x4^15*z^33 - x1^65*x2^50*x3^32*x4^13*z^32 + x1^64*x2^51*x3^32*x4^13*z^32 + x1^65*x2^50*x3^31*x4^14*z^32 - x1^64*x2^50*x3^32*x4^14*z^32 - x1^63*x2^51*x3^32*x4^14*z^32 + 2*x1^62*x2^51*x3^33*x4^14*z^32 - x1^64*x2^49*x3^32*x4^15*z^32 + x1^63*x2^50*x3^32*x4^15*z^32 + x1^62*x2^51*x3^32*x4^15*z^32 - 2*x1^62*x2^50*x3^33*x4^15*z^32 - 2*x1^61*x2^51*x3^33*x4^15*z^32 + x1^63*x2^48*x3^34*x4^15*z^32 + x1^62*x2^49*x3^34*x4^15*z^32 + x1^60*x2^51*x3^34*x4^15*z^32 - x1^64*x2^49*x3^30*x4^12*z^31 + x1^63*x2^49*x3^31*x4^12*z^31 - x1^62*x2^50*x3^31*x4^12*z^31 - x1^63*x2^48*x3^31*x4^13*z^31 - x1^62*x2^49*x3^31*x4^13*z^31 + 2*x1^61*x2^50*x3^31*x4^13*z^31 + x1^62*x2^48*x3^32*x4^13*z^31 + x1^61*x2^49*x3^32*x4^13*z^31 - 2*x1^60*x2^50*x3^32*x4^13*z^31 + x1^62*

In [24]:
out

-x1^58*x2^44*x3^18*z^24 - x1^56*x2^42*x3^17*z^23 + x1^55*x2^42*x3^18*z^23 + x1^54*x2^43*x3^18*z^23 + x1^54*x2^40*x3^16*z^22 + x1^52*x2^41*x3^17*z^22 + x1^51*x2^42*x3^17*z^22 + x1^53*x2^39*x3^18*z^22 - x1^52*x2^40*x3^18*z^22 - x1^50*x2^42*x3^18*z^22 + x1^53*x2^39*x3^13*z^21 + x1^52*x2^38*x3^15*z^21 - x1^51*x2^38*x3^16*z^21 - x1^50*x2^39*x3^16*z^21 + 2*x1^51*x2^37*x3^17*z^21 - x1^48*x2^40*x3^17*z^21 - x1^50*x2^37*x3^18*z^21 - x1^49*x2^38*x3^18*z^21 + x1^48*x2^39*x3^18*z^21 - x1^49*x2^38*x3^13*z^20 - x1^48*x2^39*x3^13*z^20 - x1^50*x2^36*x3^14*z^20 + x1^47*x2^39*x3^14*z^20 - x1^48*x2^37*x3^15*z^20 - x1^47*x2^38*x3^15*z^20 - x1^49*x2^35*x3^16*z^20 + x1^48*x2^36*x3^16*z^20 + x1^46*x2^38*x3^16*z^20 - x1^48*x2^35*x3^17*z^20 - 2*x1^47*x2^36*x3^17*z^20 - x1^46*x2^37*x3^17*z^20 + 2*x1^47*x2^35*x3^18*z^20 - x1^46*x2^36*x3^18*z^20 + x1^45*x2^37*x3^18*z^20 - x1^49*x2^36*x3^10*z^19 - x1^49*x2^35*x3^11*z^19 + x1^48*x2^36*x3^11*z^19 - x1^47*x2^37*x3^11*z^19 - 2*x1^48*x2^34*x3^13*z^19 + x1^47*x2^35*x3^1

In [20]:
factor(den_guess())

(-1) * (x1^2*x2^2*x3*z - 1) * (x1^2*x2^2*x3*z + 1) * (x1^3*x2*x3*z - 1) * (x1^3*x2*x3*z + 1) * (x1^4*x2*z - 1) * (x1^5*z - 1) * (x1^6*x2^2*x3^2*z^2 + 1) * (x1^5*x2^5*z^2 + 1) * (x1^5*x2^5*z^2 - 1)^2 * (x1^5*x2^5*x3^5*z^3 - 1) * (x1^10*x2^10*z^4 + 1) * (x1^10*x2^10*x3^10*z^6 + 1)

In [21]:
out/den_guess()

(-x1^45*x2^33*x3^17*z^19 - x1^43*x2^31*x3^16*z^18 - x1^42*x2^32*x3^16*z^18 + x1^42*x2^31*x3^17*z^18 + x1^41*x2^32*x3^17*z^18 + x1^41*x2^29*x3^15*z^17 - x1^40*x2^30*x3^15*z^17 - x1^39*x2^31*x3^15*z^17 + 2*x1^39*x2^30*x3^16*z^17 + 2*x1^38*x2^31*x3^16*z^17 - x1^40*x2^28*x3^17*z^17 - x1^39*x2^29*x3^17*z^17 - x1^37*x2^31*x3^17*z^17 + x1^40*x2^28*x3^12*z^16 + x1^39*x2^27*x3^14*z^16 + x1^38*x2^28*x3^14*z^16 - x1^37*x2^29*x3^14*z^16 - x1^36*x2^30*x3^14*z^16 - x1^38*x2^27*x3^15*z^16 - x1^37*x2^28*x3^15*z^16 + 2*x1^36*x2^29*x3^15*z^16 + 2*x1^35*x2^30*x3^15*z^16 - x1^37*x2^27*x3^16*z^16 - x1^36*x2^28*x3^16*z^16 - x1^35*x2^29*x3^16*z^16 - x1^34*x2^30*x3^16*z^16 + x1^37*x2^26*x3^17*z^16 + x1^36*x2^27*x3^17*z^16 + x1^35*x2^28*x3^17*z^16 + x1^37*x2^27*x3^11*z^15 - x1^36*x2^27*x3^12*z^15 - x1^35*x2^28*x3^12*z^15 - x1^37*x2^25*x3^13*z^15 + x1^36*x2^26*x3^13*z^15 + x1^35*x2^27*x3^13*z^15 - x1^33*x2^29*x3^13*z^15 - 2*x1^35*x2^26*x3^14*z^15 - 2*x1^34*x2^27*x3^14*z^15 + 2*x1^33*x2^28*x3^14*z^15 + 2*x1^32*x

In [44]:
nvars = 1
target_partition = [4,1]

BR = QQ[",".join("x"+str(i) for i in range(1, nvars+1))+",z"]
BR.inject_variables()
SymmetricFunctions(QQ).inject_shorthands(verbose=False)

def do_P_k(k, n):
    global BR, nvars
    CR = s[1].expand(nvars).parent()
    return s[n].expand(nvars).subs({CR.gens()[i] : BR('x'+str(i+1))**k for i in range(nvars)})

@cached_function
def do_P_lambda(la, n):
    global BR, nvars
    a = BR(expand(mul(do_P_k(p, n) for p in la) * mul(BR('x'+str(i+1)) - BR('x'+str(j+1)) for i in range(nvars) for j in range(i+1, nvars))))
    return sum(c * mul(BR('x'+str(i+1))^(v[i] - (nvars-i-1)) for i in range(nvars)) \
               for (v,c) in a.dict().items() if all(v[i] > v[i+1] for i in range(nvars-1)))

@cached_function
def den_coeff(d):
    global BR
    gz = BR.gens()[-1]
    return BR(SR(den_guess()).coefficient(gz, d))

def calc_num(la, d):
    return sum(den_coeff(d-r) * do_P_lambda(Partition(la), r) for r in range(d+1))

@cached_function
def den_guess():
    global BR
    return BR(1-z*x1**5)

out = 0
for d in range(0, 20):
    CC = calc_num(target_partition, d)
    if CC:
        print(d, len(list(CC)), list(CC)[:3])
    else:
        print(d, 0, "[]")
    
    out += z**d * CC

Defining x1, z
0 1 [(1, 1)]
1 0 []
2 0 []
3 0 []
4 0 []
5 0 []
6 0 []
7 0 []
8 0 []
9 0 []
10 0 []
11 0 []
12 0 []
13 0 []
14 0 []
15 0 []
16 0 []
17 0 []
18 0 []
19 0 []


In [45]:
out/den_guess()

1/(-x1^5*z + 1)

The following code can be used to compute
## $${\tilde P}_\mu(x_1, x_2, \ldots, x_d;z) = \sum_{n \geq 0} p_\mu[s_n](x_1,x_2,\ldots,x_d)  z^n$$

There are two problems with this approach.

1. It is incredibly slow because it relies on the Sage implementation of the `MacMahonOmega` operator.
2. Once you compute ${\tilde P}_\mu(x_1, x_2, \ldots, x_d;z)$, you still need to apply ${\mathcal L}_{X_d}$ to the expression for it to be useful for what we need.

In [18]:
Sym = SymmetricFunctions(QQ)
Sym.inject_shorthands(verbose=False)
R = PolynomialRing(QQ, 'a, b, x1, x2, x3, x4, z').fraction_field()
R.inject_variables()
x = R.gens()[2:-1]
a = R.gens()[0]
b = R.gens()[1]
z = R.gens()[-1]
def normalize_rational_function(Q):
    # Normalize denominators
    S = PolynomialRing(PolynomialRing(QQ, 'a,b'), 'x1,x2,x3,x4,z')
    K = R.fraction_field()
    den = []
    factors = Q.denominator().factor()
    scalar = factors.unit()
    for (factor, exp) in factors:
        c = S(factor).constant_coefficient()
        den.append((K(factor) / c, exp))
        scalar *= c**exp
    den = Factorization(den)
    num = Q.numerator() / scalar
    return (num, den)
def latex_fraction(X):
    if X == 0:
        return LatexExpr("0")
    num, den = normalize_rational_function(X)
    return LatexExpr(f"\\frac{{{latex(num.factor())}}}{{{latex(den)}}}")
def CT(f, g):
    # given two polynomials f(z) and g(z), compute the constant term of f(z/a) * g(a)
    Q = f.subs({z: z / (a * b)}) * g.subs({z: a * b})
    num, den = normalize_rational_function(Q)
    PTa = MacMahonOmega(a, num, den)
    CTa = prod(PTa).subs(b=0)
    return CTa
def P(k):
    return R.one() / R.prod((1 - z * xi**k) for xi in x)

Defining a, b, x1, x2, x3, x4, z


The programs above can be used to compute the following expressions *in theory* and *given enough computer time*, but I haven't found that they work in practice.

In [2]:
P11 = CT(P(1), P(1))

In [3]:
P11

(-x1^3*x2^3*x3^3*x4^3*z^6 - x1^3*x2^3*x3^2*x4^2*z^5 - x1^3*x2^2*x3^3*x4^2*z^5 - x1^2*x2^3*x3^3*x4^2*z^5 - x1^3*x2^2*x3^2*x4^3*z^5 - x1^2*x2^3*x3^2*x4^3*z^5 - x1^2*x2^2*x3^3*x4^3*z^5 + x1^3*x2^2*x3^2*x4*z^4 + x1^2*x2^3*x3^2*x4*z^4 + x1^2*x2^2*x3^3*x4*z^4 + x1^3*x2^2*x3*x4^2*z^4 + x1^2*x2^3*x3*x4^2*z^4 + x1^3*x2*x3^2*x4^2*z^4 + 3*x1^2*x2^2*x3^2*x4^2*z^4 + x1*x2^3*x3^2*x4^2*z^4 + x1^2*x2*x3^3*x4^2*z^4 + x1*x2^2*x3^3*x4^2*z^4 + x1^2*x2^2*x3*x4^3*z^4 + x1^2*x2*x3^2*x4^3*z^4 + x1*x2^2*x3^2*x4^3*z^4 - x1^2*x2^2*x3^2*z^3 + x1^3*x2*x3*x4*z^3 + x1*x2^3*x3*x4*z^3 + x1*x2*x3^3*x4*z^3 - x1^2*x2^2*x4^2*z^3 - x1^2*x3^2*x4^2*z^3 - x2^2*x3^2*x4^2*z^3 + x1*x2*x3*x4^3*z^3 - x1^2*x2*x3*z^2 - x1*x2^2*x3*z^2 - x1*x2*x3^2*z^2 - x1^2*x2*x4*z^2 - x1*x2^2*x4*z^2 - x1^2*x3*x4*z^2 - 3*x1*x2*x3*x4*z^2 - x2^2*x3*x4*z^2 - x1*x3^2*x4*z^2 - x2*x3^2*x4*z^2 - x1*x2*x4^2*z^2 - x1*x3*x4^2*z^2 - x2*x3*x4^2*z^2 + x1*x2*z + x1*x3*z + x2*x3*z + x1*x4*z + x2*x4*z + x3*x4*z + 1)/(x1^5*x2^5*x3^5*x4^5*z^10 - x1^5*x2^5*x3^5*x4^3*z

In [ ]:
P111 = CT(P(1), P11)

In [ ]:
P1111 = CT(P111, P(1))

In [ ]:
P211 = CT(P(2), P11)

In [ ]:
P22 = CT(P(2), P(2))

In [ ]:
P31 = CT(P(3), P(1))

In [ ]:
P4 = P(4)